# 🛢️ Crude Oil Price Forecasting — Brent & WTI

**Author:** Vu Xuan Duy Anh | NEU-DSEB Year 2 | June 2026  
**GitHub:** [duyanhvuxuan25052006-lgtm](https://github.com/duyanhvuxuan25052006-lgtm)

---

## 🎯 Mục tiêu

Dự báo **giá dầu thô Brent và WTI theo tháng** cho 12 tháng tiếp theo, sử dụng:
- Dữ liệu lịch sử giá từ **Yahoo Finance** (2010 → nay)
- Các chỉ số kinh tế vĩ mô: **USD Index, S&P500, Gold, VIX**
- Pipeline: **EDA → Feature Engineering → 4 Models → Ensemble → SHAP**

## 📋 Nội dung Notebook
1. Data Ingestion
2. Exploratory Data Analysis (EDA)
3. Stationarity Test (ADF)
4. Feature Engineering
5. Walk-Forward Cross-Validation
6. Final Model Training & 12-Month Forecast
7. SHAP Explainability
8. Kết luận & Hướng phát triển

---
## 0. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import yfinance as yf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller
from prophet import Prophet
import lightgbm as lgb
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import shap

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.figsize': (13, 5), 'axes.spines.top': False, 'axes.spines.right': False})
SEED = 42
np.random.seed(SEED)

print('✅ Setup complete')

---
## 1. Data Ingestion

### 💡 Tại sao dùng dữ liệu tháng?

Giá dầu hàng ngày có **rất nhiều noise** (nhiễu ngắn hạn). Khi resample về **tháng**, ta:
- Thấy rõ **trend dài hạn** và **tính mùa vụ** (seasonality)
- Giảm noise, mô hình học ổn định hơn
- Phù hợp với bài toán **forecast kinh tế năng lượng** (người dùng quan tâm xu hướng tháng, không phải biến động giờ/ngày)

In [ ]:
# ── Tải giá dầu từ Yahoo Finance ──
print('Downloading Brent (BZ=F) and WTI (CL=F)...')

tickers = {'Brent': 'BZ=F', 'WTI': 'CL=F'}
frames = {}
for name, ticker in tickers.items():
    df = yf.download(ticker, start='2010-01-01', auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    frames[name] = df['Close'].rename(name)

daily = pd.concat(frames.values(), axis=1).dropna()
daily.index = pd.to_datetime(daily.index)

# Resample về tháng (giá đóng cửa cuối tháng)
monthly = daily.resample('ME').last()
monthly.index = monthly.index.to_period('M').to_timestamp()

print(f'Daily data : {len(daily):,} rows')
print(f'Monthly data: {len(monthly)} rows ({monthly.index[0].date()} → {monthly.index[-1].date()})')
monthly.tail()

In [ ]:
# ── Tải macro features ──
# Giải thích: các yếu tố kinh tế vĩ mô ảnh hưởng đến giá dầu
# USD mạnh → giá dầu giảm (vì dầu tính bằng USD)
# VIX cao  → bất ổn thị trường → nhu cầu dầu có thể giảm
# Gold     → safe-haven, thường biến động ngược chiều rủi ro
# S&P 500  → proxy cho tăng trưởng kinh tế → nhu cầu năng lượng

macro_tickers = {'USD_index': 'UUP', 'SP500': '^GSPC', 'Gold': 'GLD', 'VIX': '^VIX'}
macro_frames = {}
for col, ticker in macro_tickers.items():
    raw = yf.download(ticker, start='2010-01-01', auto_adjust=True, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    s = raw['Close'].resample('ME').last()
    s.index = s.index.to_period('M').to_timestamp()
    macro_frames[col] = s

macro = pd.DataFrame(macro_frames).reindex(monthly.index).ffill()
print('Macro features downloaded:', list(macro.columns))
macro.tail()

---
## 2. Exploratory Data Analysis (EDA)

### 🔑 Câu hỏi cần trả lời trong EDA:
1. Dữ liệu có đồng nhất theo thời gian không (regime change)?
2. Có tính mùa vụ rõ không?
3. Brent và WTI tương quan như thế nào?
4. Độ biến động (volatility) thay đổi theo thời gian không?

In [ ]:
# ── Fig 1: Lịch sử giá Brent & WTI ──
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

events = {
    '2014-11': ('OPEC supply glut', 'red'),
    '2020-04': ('COVID-19 crash',   'orange'),
    '2022-03': ('Russia-Ukraine',   'purple'),
}

for ax, col, color in zip(axes, ['Brent', 'WTI'], ['#0f3460', '#e94560']):
    ax.plot(monthly.index, monthly[col], color=color, lw=1.8, label=col)
    ax.fill_between(monthly.index, monthly[col], alpha=0.1, color=color)
    ax.set_ylabel('USD / barrel')
    ax.legend()
    for dt_str, (label, ec) in events.items():
        ax.axvline(pd.Timestamp(dt_str), color=ec, lw=1.2, ls='--', alpha=0.7)
        ax.text(pd.Timestamp(dt_str), ax.get_ylim()[1]*0.8, label,
                fontsize=8, color=ec, ha='center', rotation=0,
                bbox=dict(fc='white', alpha=0.7, boxstyle='round,pad=0.2'))

axes[0].set_title('Crude Oil Monthly Closing Prices (2010 – Present)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/nb_fig1_prices.png', dpi=150, bbox_inches='tight')
plt.show()

# Nhận xét:
print('\n--- NHẬN XÉT ---')
print(f'Brent peak: ${monthly["Brent"].max():.1f} (', monthly["Brent"].idxmax().strftime('%Y-%m'), ')')
print(f'Brent low : ${monthly["Brent"].min():.1f} (', monthly["Brent"].idxmin().strftime('%Y-%m'), ')')
print(f'Correlation Brent-WTI: {monthly["Brent"].corr(monthly["WTI"]):.4f}')

In [ ]:
# ── Fig 2: STL Decomposition ──
# STL = Seasonal-Trend decomposition using LOESS
# Tách chuỗi thời gian thành 3 thành phần:
#   Trend    : xu hướng dài hạn
#   Seasonal : tính mùa vụ lặp hàng năm
#   Residual : phần còn lại (noise + sự kiện bất thường)

stl = STL(monthly['Brent'].dropna(), period=12, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
labels     = ['Observed', 'Trend', 'Seasonal', 'Residual']
components = [stl.observed, stl.trend, stl.seasonal, stl.resid]
colors     = ['#1e293b', '#0f3460', '#e94560', '#64748b']

for ax, label, comp, color in zip(axes, labels, components, colors):
    ax.plot(monthly.index[-len(comp):], comp, color=color, lw=1.5)
    ax.set_ylabel(label, fontsize=10)
    if label == 'Residual':
        ax.axhline(0, color='gray', lw=0.8, ls='--')

axes[0].set_title('STL Decomposition — Brent Crude (Monthly, period=12)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/nb_fig2_stl.png', dpi=150, bbox_inches='tight')
plt.show()

# Key insight từ STL:
print('\n--- KEY INSIGHTS từ STL ---')
print('1. Trend: rõ ràng có 3 regime — tăng đến 2014, giảm mạnh, tăng lại, COVID crash 2020')
print('2. Seasonal: biên độ thay đổi theo năm → multiplicative seasonality phù hợp hơn additive')
print('3. Residual: COVID-19 (2020) và Russia-Ukraine (2022) tạo spike lớn trong residual')

In [ ]:
# ── Fig 3: Monthly Seasonality ──
monthly_copy = monthly.copy()
monthly_copy['Month']     = monthly_copy.index.month
monthly_copy['MonthName'] = monthly_copy.index.strftime('%b')
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(data=monthly_copy, x='MonthName', y='Brent', order=month_order,
            palette='Blues_r', ax=ax)
ax.set_title('Phân phối giá Brent theo tháng — Tính mùa vụ', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Brent Price (USD/barrel)')
plt.tight_layout()
plt.savefig('output/nb_fig3_seasonality.png', dpi=150, bbox_inches='tight')
plt.show()

# Tính seasonality index
seasonal_means = monthly_copy.groupby('Month')['Brent'].mean()
overall_mean   = monthly_copy['Brent'].mean()
seasonal_idx   = (seasonal_means / overall_mean * 100).round(1)
print('\nSeasonal Index (100 = average):')
print(pd.DataFrame({'Month': month_order, 'Index': seasonal_idx.values}).to_string(index=False))

In [ ]:
# ── Fig 4: Correlation Heatmap ──
# Mối quan hệ giữa giá dầu và các macro indicators

combined = pd.concat([monthly, macro], axis=1).dropna()

fig, ax = plt.subplots(figsize=(8, 6))
corr = combined.corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, square=True,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix: Oil Prices & Macro Indicators', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('output/nb_fig4_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCorrelation với Brent:')
print(combined.corr()['Brent'].sort_values(ascending=False).to_string())

---
## 3. Stationarity Test — ADF Test

### 💡 Tại sao cần kiểm tra tính dừng?

**SARIMA** yêu cầu chuỗi thời gian phải **dừng** (stationary) — tức là mean và variance không thay đổi theo thời gian.  
Nếu không dừng → phải lấy **sai phân** (differencing).

**ADF Test** (Augmented Dickey-Fuller):
- H₀: Chuỗi **không dừng** (có unit root)
- H₁: Chuỗi **dừng**
- Nếu p-value < 0.05 → bác bỏ H₀ → chuỗi dừng

In [ ]:
def adf_test(series, name='Series'):
    result = adfuller(series.dropna())
    p = result[1]
    adf_stat = result[0]
    status = 'Stationary ✅' if p < 0.05 else 'Non-stationary ⚠️'
    print(f'{name:<30} | ADF={adf_stat:7.3f} | p={p:.4f} | {status}')
    return p < 0.05

print('='*70)
print(f'{"Series":<30} | {"ADF stat":>10} | {"p-value":>8} | Status')
print('='*70)

# Test level
adf_test(monthly['Brent'],             'Brent (level)')
adf_test(monthly['WTI'],               'WTI (level)')

# Test 1st difference
adf_test(monthly['Brent'].diff().dropna(), 'Brent (1st diff)')
adf_test(monthly['WTI'].diff().dropna(),   'WTI (1st diff)')
print('='*70)
print('\nKết luận: Brent và WTI là I(1) — tích hợp bậc 1')
print('=> SARIMA nên dùng d=1 (lấy sai phân 1 lần)')

---
## 4. Feature Engineering

### 🔧 5 nhóm feature

| Nhóm | Features | Lý do |
|---|---|---|
| **Lag** | lag1, lag2, lag3, lag6, lag12 | Autocorrelation — giá tháng trước ảnh hưởng tháng này |
| **Rolling** | ma3, ma6, ma12, std3, std6 | Xu hướng ngắn-trung hạn, độ biến động |
| **Momentum** | ret1, ret3, ret12 | Đà tăng/giảm — momentum effect |
| **Calendar** | month_sin/cos, qtr_sin/cos | Mùa vụ — dùng sin/cos tránh discontinuity tháng 12→1 |
| **Macro** | USD_index, SP500, Gold, VIX + lags | Yếu tố kinh tế vĩ mô |

In [ ]:
def build_features(df, macro=None):
    feat = df.copy()
    for col in ['Brent', 'WTI']:
        if col not in feat.columns: continue
        s = feat[col]
        # Lag
        for lag in [1, 2, 3, 6, 12]:
            feat[f'{col}_lag{lag}'] = s.shift(lag)
        # Rolling
        for w in [3, 6, 12]:
            feat[f'{col}_ma{w}']  = s.shift(1).rolling(w).mean()
            feat[f'{col}_std{w}'] = s.shift(1).rolling(w).std()
        # Momentum
        feat[f'{col}_ret1']  = s.pct_change(1)
        feat[f'{col}_ret3']  = s.pct_change(3)
        feat[f'{col}_ret12'] = s.pct_change(12)

    if 'Brent' in feat and 'WTI' in feat:
        feat['spread']      = feat['Brent'] - feat['WTI']
        feat['spread_lag1'] = feat['spread'].shift(1)

    # Calendar (sin/cos để biểu diễn tuần hoàn liên tục)
    feat['month']     = feat.index.month
    feat['quarter']   = feat.index.quarter
    feat['year']      = feat.index.year
    feat['month_sin'] = np.sin(2 * np.pi * feat.index.month / 12)
    feat['month_cos'] = np.cos(2 * np.pi * feat.index.month / 12)
    feat['qtr_sin']   = np.sin(2 * np.pi * feat.index.quarter / 4)
    feat['qtr_cos']   = np.cos(2 * np.pi * feat.index.quarter / 4)

    # Regime
    feat['regime_pre_covid']  = (feat['year'] < 2020).astype(int)
    feat['regime_covid']      = ((feat['year'] >= 2020) & (feat['year'] <= 2021)).astype(int)
    feat['regime_post_covid'] = (feat['year'] >= 2022).astype(int)

    if macro is not None:
        for col in macro.columns:
            feat[col]          = macro[col]
            feat[f'{col}_lag1'] = macro[col].shift(1)
            feat[f'{col}_ret1'] = macro[col].pct_change(1)

    return feat

df_feat = build_features(monthly, macro=macro)
df_feat = df_feat.dropna(subset=['Brent'])

print(f'Feature matrix shape: {df_feat.shape}')
print(f'Total features: {df_feat.shape[1]} columns')
print('\nSample of features:')
df_feat[['Brent', 'Brent_lag1', 'Brent_ma3', 'month_sin', 'USD_index']].head()

---
## 5. Walk-Forward Cross-Validation

### 💡 Tại sao KHÔNG dùng random split?

Dữ liệu thời gian có **thứ tự quan trọng** — thông tin tương lai không được dùng để predict quá khứ.  
Nếu random split → **data leakage** → kết quả đẹp trên CV nhưng sai ngoài thực tế.

**Walk-Forward CV** đảm bảo:
- Train luôn ở **trước** test về thời gian
- Mô phỏng **thực tế deployment**: mỗi tháng train lại, dự báo tháng tiếp theo

```
Fold 1: [=TRAIN (2010-2014)=]  [TEST 6 tháng]
Fold 2: [==TRAIN (2010-2017)==]  [TEST 6 tháng]
Fold 3: [===TRAIN (2010-2019)===]  [TEST 6 tháng]
Fold 4: [====TRAIN (2010-2021)====]  [TEST 6 tháng]
Fold 5: [=====TRAIN (2010-2023)=====]  [TEST 6 tháng]
```

In [ ]:
TARGET = 'Brent'
N_FOLDS  = 5
HORIZON  = 6   # predict 6 months ahead per fold
MIN_TRAIN = 60  # at least 5 years

feature_cols = [c for c in df_feat.columns
                if c not in ['Brent', 'WTI', 'spread']
                and not c.startswith('Brent_ret') and not c.startswith('WTI_ret')]

n    = len(df_feat)
step = (n - MIN_TRAIN - HORIZON) // N_FOLDS

cv_results = []

for fold in range(N_FOLDS):
    cutoff    = MIN_TRAIN + fold * step
    train     = df_feat.iloc[:cutoff]
    test      = df_feat.iloc[cutoff:cutoff + HORIZON]
    if len(test) == 0: break

    y_train = train[TARGET].values
    y_test  = test[TARGET].values
    X_train = train[feature_cols].fillna(method='ffill').fillna(0)
    X_test  = test[feature_cols].fillna(method='ffill').fillna(0)

    scaler  = StandardScaler()
    Xtr_s   = scaler.fit_transform(X_train)
    Xte_s   = scaler.transform(X_test)

    preds = {}

    # SARIMA(1,1,1)(1,1,0,12)
    try:
        sm = SARIMAX(y_train, order=(1,1,1), seasonal_order=(1,1,0,12),
                     enforce_stationarity=False).fit(disp=False)
        preds['SARIMA'] = sm.forecast(HORIZON)
    except:
        preds['SARIMA'] = np.full(HORIZON, y_train[-1])

    # Prophet
    try:
        pdf = pd.DataFrame({'ds': train.index, 'y': y_train})
        mp  = Prophet(yearly_seasonality=True, seasonality_mode='multiplicative',
                      changepoint_prior_scale=0.05)
        mp.fit(pdf)
        fc  = mp.make_future_dataframe(periods=HORIZON, freq='ME')
        preds['Prophet'] = mp.predict(fc)['yhat'].values[-HORIZON:]
    except:
        preds['Prophet'] = np.full(HORIZON, y_train[-1])

    # XGBoost
    xgb_m = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05,
                               max_depth=4, subsample=0.8, random_state=SEED, verbosity=0)
    xgb_m.fit(Xtr_s, y_train)
    preds['XGBoost'] = xgb_m.predict(Xte_s)

    # LightGBM
    lgb_m = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31,
                                subsample=0.8, random_state=SEED, verbosity=-1)
    lgb_m.fit(Xtr_s, y_train)
    preds['LightGBM'] = lgb_m.predict(Xte_s)

    # Ensemble
    preds['Ensemble'] = 0.5*preds['LightGBM'] + 0.3*preds['XGBoost'] + 0.2*preds['Prophet']

    row = {'fold': fold+1, 'cutoff': df_feat.index[cutoff-1].strftime('%Y-%m'),
           'n_train': len(train)}
    for mdl, pred in preds.items():
        row[f'{mdl}_MAE']  = round(mean_absolute_error(y_test, pred), 2)
        row[f'{mdl}_MAPE'] = round(np.mean(np.abs((y_test-pred)/(y_test+1e-8)))*100, 2)

    cv_results.append(row)
    print(f'Fold {fold+1} (cutoff {row["cutoff"]}): LGB MAE={row["LightGBM_MAE"]:.2f} | Ensemble MAE={row["Ensemble_MAE"]:.2f}')

cv_df = pd.DataFrame(cv_results)
cv_df.to_csv('output/cv_results.csv', index=False)

In [ ]:
# ── CV Summary Table ──
models = ['SARIMA', 'Prophet', 'XGBoost', 'LightGBM', 'Ensemble']
summary_rows = []
for m in models:
    summary_rows.append({
        'Model':     m,
        'Mean MAE':  cv_df[f'{m}_MAE'].mean().round(2),
        'Std MAE':   cv_df[f'{m}_MAE'].std().round(2),
        'Mean MAPE': cv_df[f'{m}_MAPE'].mean().round(2),
    })
summary = pd.DataFrame(summary_rows).set_index('Model').sort_values('Mean MAE')
print('\n=== Walk-Forward CV Results (5 folds, horizon=6) ===')
print(summary.to_string())
print('\n=> LightGBM là model tốt nhất!')
print('=> Prophet kém vì cú shock COVID gây structural break → trend extrapolation sai')

---
## 6. Final Model Training & 12-Month Forecast

Train trên **toàn bộ dữ liệu lịch sử**, forecast 12 tháng tới.

In [ ]:
y_all  = df_feat[TARGET].values
X_all  = df_feat[feature_cols].fillna(method='ffill').fillna(0)
scaler = StandardScaler()
X_s    = scaler.fit_transform(X_all)

# Build future dates
HORIZON_FINAL = 12
future_dates  = pd.date_range(df_feat.index[-1] + pd.DateOffset(months=1),
                               periods=HORIZON_FINAL, freq='MS')

# Build future feature matrix (calendar + last known values)
future_feat = pd.DataFrame(index=future_dates)
future_feat['month']     = future_dates.month
future_feat['quarter']   = future_dates.quarter
future_feat['year']      = future_dates.year
future_feat['month_sin'] = np.sin(2*np.pi*future_dates.month/12)
future_feat['month_cos'] = np.cos(2*np.pi*future_dates.month/12)
future_feat['qtr_sin']   = np.sin(2*np.pi*future_dates.quarter/4)
future_feat['qtr_cos']   = np.cos(2*np.pi*future_dates.quarter/4)
future_feat['regime_pre_covid']  = 0
future_feat['regime_covid']      = 0
future_feat['regime_post_covid'] = 1
for col in feature_cols:
    if col not in future_feat.columns:
        future_feat[col] = X_all[col].iloc[-1]  # use last known value

X_future = scaler.transform(future_feat[feature_cols].fillna(0))

# Train final LightGBM (best model)
lgb_final = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03,
                                num_leaves=31, subsample=0.8, random_state=SEED, verbosity=-1)
lgb_final.fit(X_s, y_all)

# Train final XGBoost
xgb_final = xgb.XGBRegressor(n_estimators=500, learning_rate=0.03,
                               max_depth=4, subsample=0.8, random_state=SEED, verbosity=0)
xgb_final.fit(X_s, y_all)

# Train Prophet
pdf_all = pd.DataFrame({'ds': df_feat.index, 'y': y_all})
mp_final = Prophet(yearly_seasonality=True, seasonality_mode='multiplicative',
                   changepoint_prior_scale=0.05, interval_width=0.95)
mp_final.fit(pdf_all)
future_p = mp_final.make_future_dataframe(periods=HORIZON_FINAL, freq='ME')
fc_p     = mp_final.predict(future_p).tail(HORIZON_FINAL)

# Ensemble (weighted)
lgb_pred     = lgb_final.predict(X_future)
xgb_pred     = xgb_final.predict(X_future)
prophet_pred = fc_p['yhat'].values
ensemble     = 0.5*lgb_pred + 0.3*xgb_pred + 0.2*prophet_pred

fc_df = pd.DataFrame({
    'Date':     future_dates,
    'LightGBM': lgb_pred,
    'XGBoost':  xgb_pred,
    'Prophet':  prophet_pred,
    'Ensemble': ensemble,
    'CI_lower': fc_p['yhat_lower'].values,
    'CI_upper': fc_p['yhat_upper'].values,
}).set_index('Date')

fc_df.to_csv('output/forecast_12months.csv')
print('12-Month Brent Forecast:')
print(fc_df[['Ensemble','CI_lower','CI_upper']].to_string())

In [ ]:
# ── Forecast Plot (Plotly Interactive) ──
hist_tail = monthly[TARGET].iloc[-48:]

fig = go.Figure()
fig.add_trace(go.Scatter(x=hist_tail.index, y=hist_tail.values,
                          name='Historical', line=dict(color='#1e293b', width=2.5)))
fig.add_trace(go.Scatter(x=fc_df.index, y=fc_df['Ensemble'],
                          name='Ensemble Forecast',
                          line=dict(color='#e94560', width=2.5, dash='dash')))
fig.add_trace(go.Scatter(
    x=list(fc_df.index) + list(fc_df.index[::-1]),
    y=list(fc_df['CI_upper']) + list(fc_df['CI_lower'][::-1]),
    fill='toself', fillcolor='rgba(233,69,96,0.12)',
    line=dict(color='rgba(0,0,0,0)'), name='95% CI'))
for mdl, color in [('LightGBM','#0f3460'), ('XGBoost','#475569'), ('Prophet','#94a3b8')]:
    fig.add_trace(go.Scatter(x=fc_df.index, y=fc_df[mdl], name=mdl,
                              line=dict(color=color, width=1.5, dash='dot'), opacity=0.7))

fig.update_layout(
    title='Brent Crude Oil — 12-Month Ensemble Forecast',
    xaxis_title='Date', yaxis_title='USD / barrel',
    template='plotly_white', hovermode='x unified',
    legend=dict(orientation='h', y=1.02, xanchor='right', x=1),
    font=dict(family='Inter, sans-serif', size=12)
)
fig.write_html('output/forecast_interactive.html')
fig.show()

print('Interactive chart saved to output/forecast_interactive.html')

---
## 7. SHAP Explainability

### 💡 SHAP là gì?

**SHAP (SHapley Additive exPlanations)** giải thích **tại sao** model đưa ra dự báo cụ thể:
- Mỗi feature được gán một **SHAP value** = mức ảnh hưởng đến prediction
- Positive SHAP → feature này **đẩy prediction lên**
- Negative SHAP → feature này **kéo prediction xuống**

Đây là công cụ **explainable AI** rất quan trọng trong bài toán tài chính/năng lượng — stakeholder cần hiểu tại sao model dự báo như vậy.

In [ ]:
explainer   = shap.TreeExplainer(lgb_final)
shap_values = explainer.shap_values(X_s)

# Bar plot: top 15 features by mean |SHAP|
fig, ax = plt.subplots(figsize=(9, 7))
shap.summary_plot(shap_values, X_s,
                  feature_names=feature_cols,
                  plot_type='bar',
                  max_display=15,
                  show=False)
plt.title('SHAP Feature Importance — LightGBM (Brent Forecast)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/nb_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

# Top features
mean_shap = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({'Feature': feature_cols, 'Mean |SHAP|': mean_shap})
shap_df = shap_df.sort_values('Mean |SHAP|', ascending=False).head(10)
print('Top 10 features by SHAP importance:')
print(shap_df.to_string(index=False))

---
## 8. Kết luận & Hướng Phát Triển

### 📌 Key Findings

1. **Giá dầu là I(1)** — cần differencing trước khi dùng SARIMA
2. **LightGBM outperforms** SARIMA và Prophet (MAE $7.45 vs $22.88 của Prophet)
3. **Prophet kém** do COVID-19 gây structural break — Prophet overfit trend trước 2020
4. **Lag features** (đặc biệt lag1, lag3) là quan trọng nhất theo SHAP
5. **Macro features** (USD_index, Gold) đóng vai trò đáng kể trong prediction
6. **Ensemble** không phải lúc nào cũng tốt hơn model đơn lẻ tốt nhất — LightGBM single > Ensemble

### 🚀 Hướng Phát Triển

- **OPEC+ production data**: thêm dữ liệu sản lượng khai thác thực tế
- **Geopolitical risk index**: news sentiment analysis
- **LSTM / Transformer**: deep learning cho long-range dependencies
- **Probabilistic forecasting**: Bayesian deep learning cho uncertainty quantification
- **Multi-step ahead**: recursive vs. direct strategy comparison

In [ ]:
print('='*60)
print(' PROJECT SUMMARY')
print('='*60)
print(f'Data        : {len(monthly)} monthly observations (2010-{monthly.index[-1].year})')
print(f'Features    : {len(feature_cols)} engineered features')
print(f'Best model  : LightGBM (Mean MAE={summary.loc["LightGBM","Mean MAE"]} USD/barrel)')
print(f'Forecast    : 12 months ahead ({future_dates[0].strftime("%Y-%m")} to {future_dates[-1].strftime("%Y-%m")})')
print(f'Brent range : ${fc_df["Ensemble"].min():.1f} - ${fc_df["Ensemble"].max():.1f} /barrel')
print('='*60)
print('Outputs saved in ./output/')